# First look at `gambling.csv` — cleaning + saving something usable

I got this export from my poker scrape / history dump (`data/gambling.csv`). Before touching models I want to actually **see** what’s in it: column names, weird encoding, how often fields are missing, and whether `result` / `balance` look like a sane label for "did this seat win money on the hand".

Later I bolt on a few **custom columns** (streaks, rough strength proxy, bluff heuristic, fake "equity" for preflop — same helpers the project code uses). The important bit: I **don’t** dump z-scored duplicate copies of every street feature into the CSV. The poker simulator recomputes card/board features from the raw columns so training matches the game (`model_train.py` ↔ UI).

**End of this notebook:** write `data/cleanedGambling.csv` + a slim `artifacts/processed/model_ready.csv` for the Streamlit EDA tab.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from scripts.data.training_dataset import (
    describe_training_alignment,
    export_cleaned_gambling_csv,
    export_model_ready_csv,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)

RAW_PATH = ROOT / "data" / "gambling.csv"
print("Using:", RAW_PATH)

## Load it like I’m opening a CSV for the first time

In [ ]:
df_raw = pd.read_csv(RAW_PATH)
print("shape:", df_raw.shape)
df_raw.head(8)

In [ ]:
# dtypes + memory-ish overview
df_raw.info()

### What I’m looking at (informally)
- **IDs / context:** `hand_id`, `tourn_id`, `table`, `date`, `time`, `seat`, `table_size` — lets me group rows and sort in time.
- **Cards:** `cards`, `board_flop`, `board_turn`, `board_river` — messy strings (`--`, `0`, spaced ranks). Parsing happens in code, not by eye.
- **Money:** `pot_*`, `bet_*`, `stack`, `blinds`, `balance` — numeric-ish but sometimes strings with `$` in other datasets; here mostly plain numbers.
- **Outcome:** `result` text (`took chips`, `gave up`, …) + `balance` chip delta — that’s what I’ll treat as **won vs lost** for supervised stuff.
- **Actions:** `action_pre`, … — text used later for aggression / bluff-ish labels.

In [ ]:
# Numeric summaries for anything that already parses as a number
num_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()
df_raw[num_cols].describe().T.head(20)

In [ ]:
# How often does each "result" show up?
df_raw["result"].value_counts(dropna=False).head(15)

In [ ]:
# Rough sense of label balance (I'll formalize won_flag in the pipeline)
r = df_raw["result"].astype(str).str.lower()
won = r.str.contains("took chips|won", regex=True)
lost = r.str.contains("lost|gave up", regex=True)
pd.Series({"won-ish": won.mean(), "lost-ish": lost.mean(), "other": (~(won | lost)).mean()})

In [ ]:
# Spot check — hidden / mucked hole cards (normal when someone folds pre)
c = df_raw["cards"].astype(str).str.strip().str.lower()
mucked = c.isin(["--", ""]) | c.eq("nan")
print("fraction rows without visible hole cards:", round(float(mucked.mean()), 4))

## Missing values — what breaks first?
No fancy imputation story yet: I mostly want to know if any column is empty half the time (usually board columns early streets, or optional fields).

In [ ]:
miss = (df_raw.isna().mean().sort_values(ascending=False) * 100).head(25)
miss.to_frame("missing_pct")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
miss.iloc[::-1].plot(kind="barh", ax=ax, color="#486581")
ax.set_xlabel("Missing %")
ax.set_title("Columns sorted by missingness (top 25)")
plt.tight_layout()
plt.show()

## A couple plots just to feel the distribution
Balance is my quick proxy for "did money move"; table size tells me if I’m mostly looking at short-handed spin-type tables vs bigger ones.

In [ ]:
bal = pd.to_numeric(df_raw.get("balance"), errors="coerce")
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(bal.dropna(), bins=80, color="#829AB1", edgecolor="white")
axes[0].set_title("Hand balance (numeric)")
axes[0].set_xlabel("balance")
if "table_size" in df_raw.columns:
    df_raw["table_size"].fillna(-1).astype(int).value_counts().sort_index().plot(
        kind="bar", ax=axes[1], color="#334E68"
    )
    axes[1].set_title("Table size (seat count)")
plt.tight_layout()
plt.show()

## Custom features + export (I lean on shared project code here)
I’m not going to hand-roll everything in the notebook — the repo already has `scripts/data/training_dataset.py` so my cleaned file stays consistent with:
- **`model_train.py`** (stage win models — features rebuilt from cards/pots/bets)
- **Dashboard models** (win / bluff / money — tabular features like streaks + aggression)

Concrete additions you’ll see in `cleanedGambling.csv`:
- `player_id` hashed from `name` (I drop raw screen names)
- `hand_datetime` from `date` + `time`
- `net_result`, `won_flag`, `starting_stack`, encoded `table_position`
- streak features, deterministic strength bands, bluff heuristic, `preflop_equity` proxy

If you’re grading explainability: **street-level card features are NOT duplicated here as hundreds of normalized columns** — that would drift from how the game computes features live.

In [ ]:
cleaned_path = export_cleaned_gambling_csv(RAW_PATH, ROOT / "data" / "cleanedGambling.csv")
cleaned = pd.read_csv(cleaned_path)
model_ready_path = export_model_ready_csv(cleaned, ROOT / "artifacts" / "processed" / "model_ready.csv")

print("Wrote:", cleaned_path)
print("Wrote:", model_ready_path)
print("cleaned shape:", cleaned.shape)
cleaned.head(4)

In [ ]:
# Peek at a few engineered columns I care about downstream
peek = [
    c
    for c in [
        "won_flag",
        "net_result",
        "starting_stack",
        "preflop_equity",
        "win_streak",
        "loss_streak",
        "aggression_score",
        "strength_mean",
        "is_bluffing",
    ]
    if c in cleaned.columns
]
cleaned[peek].describe().T

In [ ]:
print(describe_training_alignment())

### Next step for me
- Open **`02_eda_stage_models_visual.ipynb`** — I’ll sanity-check which **street-level** inputs actually move a RandomForest trained the same way as `model_train.py`, then compare against the **reduced** lists in `scripts/models/feature_contracts.py`.
- Then **`03_visible_bluff_model.ipynb`** — observable-only bluff features + importance snapshot (production training: `python visible_bluff_train.py`).
- After I’m happy: **`python model_train.py`** from the project root (reads `cleanedGambling.csv` if it exists) so **`feature_names.pkl`** matches the shorter stage feature sets.